### Final Silver Dataset Output

At this stage the Silver dataset has completed all cleaning and transformation steps.  
The dataset now contains standardized schema, decoded categorical variables, and derived analytical metrics.

Key transformations applied in the Silver layer include:

- Selection of core analytical columns from the raw College Scorecard dataset
- Conversion of numeric fields to appropriate data types
- Decoding of categorical variables (`CONTROL` → `CONTROL_DESC`)
- Grouping of locale classifications (`LOCALE` → `LOCALE_GROUP`)
- Creation of derived metrics:
  - `AVG_TUITION`
  - `EARNINGS_TO_TUITION_RATIO`
- Introduction of a data quality indicator (`ANALYTICS_READY`) to identify records with sufficient information for analysis

The resulting dataset contains 6429 institutions and 16 curated columns and is now ready for downstream analytical modeling in the Gold layer.

The cleaned dataset is saved to the `data_processed` directory for use in subsequent notebooks.

In [2]:
import pandas as pd

silver_df = pd.read_csv("../data_processed/silver_college_scorecard.csv", low_memory=False)

print(silver_df.shape)
silver_df.head()

(6429, 16)


,UNITID,INSTNM,CITY,STABBR,CONTROL,LOCALE,UGDS,TUITIONFEE_IN,TUITIONFEE_OUT,C150_4,MD_EARN_WNE_P10,CONTROL_DESC,LOCALE_GROUP,AVG_TUITION,EARNINGS_TO_TUITION_RATIO,ANALYTICS_READY
0,100654,Alabama A & M University,Normal,AL,1,12.0,5726.0,10024.0,18634.0,0.2874,40628.0,Public,City,14329.0,2.835369,True
1,100663,University of Alabama at Birmingham,Birmingham,AL,1,12.0,12118.0,8832.0,21864.0,0.6260,54501.0,Public,City,15348.0,3.551016,True
2,100690,Amridge University,Montgomery,AL,2,12.0,226.0,NaN,NaN,0.4000,37621.0,Private nonprofit,City,NaN,NaN,False
3,100706,University of Alabama in Huntsville,Huntsville,AL,1,12.0,6650.0,11770.0,24662.0,0.6191,61767.0,Public,City,18216.0,3.390810,True
4,100724,Alabama State University,Montgomery,AL,1,12.0,3322.0,11248.0,19576.0,0.3018,34502.0,Public,City,15412.0,2.238645,True


### Filter to analytics ready records:

In [3]:
gold_base_df = silver_df[silver_df["ANALYTICS_READY"] == True].copy()

print(gold_base_df.shape)
gold_base_df.head()

(2057, 16)


,UNITID,INSTNM,CITY,STABBR,CONTROL,LOCALE,UGDS,TUITIONFEE_IN,TUITIONFEE_OUT,C150_4,MD_EARN_WNE_P10,CONTROL_DESC,LOCALE_GROUP,AVG_TUITION,EARNINGS_TO_TUITION_RATIO,ANALYTICS_READY
0,100654,Alabama A & M University,Normal,AL,1,12.0,5726.0,10024.0,18634.0,0.2874,40628.0,Public,City,14329.0,2.835369,True
1,100663,University of Alabama at Birmingham,Birmingham,AL,1,12.0,12118.0,8832.0,21864.0,0.6260,54501.0,Public,City,15348.0,3.551016,True
3,100706,University of Alabama in Huntsville,Huntsville,AL,1,12.0,6650.0,11770.0,24662.0,0.6191,61767.0,Public,City,18216.0,3.390810,True
4,100724,Alabama State University,Montgomery,AL,1,12.0,3322.0,11248.0,19576.0,0.3018,34502.0,Public,City,15412.0,2.238645,True
5,100751,The University of Alabama,Tuscaloosa,AL,1,12.0,32323.0,11900.0,33200.0,0.7369,59221.0,Public,City,22550.0,2.626208,True


## Gold Table 1: Institution ROI Ranking

This table ranks institutions using tuition and earnings metrics. It supports analyses of educational value and return on investment.

In [4]:
gold_institution_roi = gold_base_df[[
    "UNITID",
    "INSTNM",
    "CITY",
    "STABBR",
    "CONTROL_DESC",
    "LOCALE_GROUP",
    "UGDS",
    "AVG_TUITION",
    "C150_4",
    "MD_EARN_WNE_P10",
    "EARNINGS_TO_TUITION_RATIO"
]].copy()

gold_institution_roi = gold_institution_roi.sort_values(
    by="EARNINGS_TO_TUITION_RATIO",
    ascending=False
)

gold_institution_roi.head(10)

,UNITID,INSTNM,CITY,STABBR,CONTROL_DESC,LOCALE_GROUP,UGDS,AVG_TUITION,C150_4,MD_EARN_WNE_P10,EARNINGS_TO_TUITION_RATIO
2202,197027,United States Merchant Marine Academy,Kings Point,NY,Public,Suburb,947.0,945.0,0.8061,90610.0,95.883598
1118,155140,Haskell Indian Nations University,Lawrence,KS,Public,City,878.0,600.0,0.4259,37043.0,61.738333
87,105297,Dine College,Tsaile,AZ,Public,Rural,1507.0,1410.0,0.0549,29188.0,20.700709
3127,226408,College of the Mainland,Texas City,TX,Public,City,3342.0,2823.0,0.2825,39639.0,14.041445
3622,247834,Collin County Community College District,McKinney,TX,Public,Suburb,25857.0,3739.0,0.2181,48701.0,13.025140
3233,230418,Ensign College,Salt Lake City,UT,Private nonprofit,City,5969.0,3888.0,0.3109,50630.0,13.022119
3070,223506,Brazosport College,Lake Jackson,TX,Public,Suburb,2827.0,3549.5,0.3867,45910.0,12.934216
2361,200527,Turtle Mountain Community College,Belcourt,ND,Private nonprofit,Rural,613.0,2626.0,0.5447,32079.0,12.215918
3162,227979,San Jacinto Community College,Pasadena,TX,Public,City,23071.0,3672.0,0.3439,43062.0,11.727124
3223,230038,Brigham Young University,Provo,UT,Private nonprofit,City,32221.0,6496.0,0.8221,75790.0,11.667180


## Gold Table 2: State Summary

This table aggregates institution-level metrics to the state level. It supports geographic comparisons of tuition, graduation rates, and earnings outcomes.

In [6]:
gold_state_summary = (
    gold_base_df
    .groupby("STABBR")
    .agg(
        institution_count=("UNITID", "count"),
        avg_tuition=("AVG_TUITION", "mean"),
        avg_grad_rate=("C150_4", "mean"),
        avg_earnings=("MD_EARN_WNE_P10", "mean"),
        avg_roi=("EARNINGS_TO_TUITION_RATIO", "mean")
    )
    .reset_index()
    .sort_values(by="avg_roi", ascending=False)
)

gold_state_summary.head(10)

,STABBR,institution_count,avg_tuition,avg_grad_rate,avg_earnings,avg_roi
28,MP,1,4779.000000,0.421600,27836.000000,5.824650
3,AS,1,5610.000000,0.477400,30087.000000,5.363102
32,ND,14,11420.142857,0.482629,47224.071429,5.343959
53,WA,56,17931.062500,0.463330,52789.964286,5.256670
11,FM,1,5050.000000,0.294700,24651.000000,4.881386
56,WY,5,9181.200000,0.417340,42790.800000,4.820173
49,UT,15,15766.633333,0.532253,56462.266667,4.804859
13,GU,3,6659.666667,0.381700,28222.666667,4.645026
4,AZ,23,17543.586957,0.354600,48660.304348,4.542558
19,KS,29,25052.431034,0.473676,50672.758621,4.348770


## Gold Table 3: Institution Type Summary

This table compares outcomes across institution sectors such as public, private nonprofit, and private for-profit schools.

In [7]:
gold_control_summary = (
    gold_base_df
    .groupby("CONTROL_DESC")
    .agg(
        institution_count=("UNITID", "count"),
        avg_tuition=("AVG_TUITION", "mean"),
        avg_grad_rate=("C150_4", "mean"),
        avg_earnings=("MD_EARN_WNE_P10", "mean"),
        avg_roi=("EARNINGS_TO_TUITION_RATIO", "mean")
    )
    .reset_index()
    .sort_values(by="avg_roi", ascending=False)
)

gold_control_summary

,CONTROL_DESC,institution_count,avg_tuition,avg_grad_rate,avg_earnings,avg_roi
2,Public,768,13937.853516,0.469827,51537.139323,4.586193
0,Private for-profit,164,18431.902439,0.432484,46991.628049,2.740415
1,Private nonprofit,1125,34376.212889,0.568857,54707.813333,1.969298


## Gold Table 4: Locale Summary

This table compares institutions by geographic environment, such as city, suburb, town, and rural settings.

In [8]:
gold_locale_summary = (
    gold_base_df
    .groupby("LOCALE_GROUP")
    .agg(
        institution_count=("UNITID", "count"),
        avg_tuition=("AVG_TUITION", "mean"),
        avg_grad_rate=("C150_4", "mean"),
        avg_earnings=("MD_EARN_WNE_P10", "mean"),
        avg_roi=("EARNINGS_TO_TUITION_RATIO", "mean")
    )
    .reset_index()
    .sort_values(by="avg_roi", ascending=False)
)

gold_locale_summary

,LOCALE_GROUP,institution_count,avg_tuition,avg_grad_rate,avg_earnings,avg_roi
1,Rural,185,22076.759459,0.422256,45343.918919,3.161111
3,Town,356,23564.905899,0.509815,50789.345506,3.041019
0,City,1023,26043.536657,0.538535,54153.639296,2.990737
2,Suburb,492,26990.747967,0.530267,54756.065041,2.957868


# Validation Checks:

## Gold Layer Validation Checks

The following checks validate that Gold-layer aggregations were created successfully and are suitable for dashboarding and reporting.

These checks confirm:
- the Gold base dataset only includes analytics-ready institutions
- summary tables aggregate correctly
- grouping dimensions are valid
- ranking outputs are reasonable for downstream visualization

#### Check 1: confirm Gold base only has analytics-ready records

In [9]:
print("Gold base shape:", gold_base_df.shape)
print("\nANALYTICS_READY distribution in Gold base:")
print(gold_base_df["ANALYTICS_READY"].value_counts(dropna=False))

Gold base shape: (2057, 16)

ANALYTICS_READY distribution in Gold base:
ANALYTICS_READY
True    2057
Name: count, dtype: int64


#### Check 2: check for null grouping fields in Gold summaries

In [10]:
print("Null STABBR in gold base:", gold_base_df["STABBR"].isna().sum())
print("Null CONTROL_DESC in gold base:", gold_base_df["CONTROL_DESC"].isna().sum())
print("Null LOCALE_GROUP in gold base:", gold_base_df["LOCALE_GROUP"].isna().sum())

Null STABBR in gold base: 0
Null CONTROL_DESC in gold base: 0
Null LOCALE_GROUP in gold base: 1


#### Check 3: verify summary table sizes

In [11]:
print("gold_institution_roi shape:", gold_institution_roi.shape)
print("gold_state_summary shape:", gold_state_summary.shape)
print("gold_control_summary shape:", gold_control_summary.shape)
print("gold_locale_summary shape:", gold_locale_summary.shape)

gold_institution_roi shape: (2057, 11)
gold_state_summary shape: (57, 6)
gold_control_summary shape: (3, 6)
gold_locale_summary shape: (4, 6)


#### Check 4: confirm ROI ranking is sorted correctly

In [12]:
gold_institution_roi["EARNINGS_TO_TUITION_RATIO"].head(10)

2202    95.883598
1118    61.738333
87      20.700709
3127    14.041445
3622    13.025140
3233    13.022119
3070    12.934216
2361    12.215918
3162    11.727124
3223    11.667180
Name: EARNINGS_TO_TUITION_RATIO, dtype: float64

In [13]:
print(
    gold_institution_roi["EARNINGS_TO_TUITION_RATIO"].is_monotonic_decreasing
)

True


#### Check 5: check for duplicate institutions in the institution Gold table

In [14]:
print("Duplicate UNITID in gold_institution_roi:",
      gold_institution_roi["UNITID"].duplicated().sum())

Duplicate UNITID in gold_institution_roi: 0


#### Check 6: descriptive stats on Gold ROI table

In [15]:
gold_institution_roi[[
    "AVG_TUITION",
    "C150_4",
    "MD_EARN_WNE_P10",
    "EARNINGS_TO_TUITION_RATIO"
]].describe()

,AVG_TUITION,C150_4,MD_EARN_WNE_P10,EARNINGS_TO_TUITION_RATIO
count,2057.000000,2057.000000,2057.000000,2057.000000
mean,25474.157997,0.521011,52908.818668,3.007819
std,16287.171183,0.204681,16322.978752,3.081676
min,600.000000,0.000000,17360.000000,0.463727
25%,13086.000000,0.379800,42187.000000,1.529667
50%,20149.000000,0.524800,51006.000000,2.481561
75%,36032.000000,0.664200,60613.000000,3.747884
max,69330.000000,1.000000,143372.000000,95.883598


---

The validation checks confirm that the Gold layer tables were generated successfully and are suitable for downstream analytics and visualization.

Key results:
- The Gold base dataset contains 2057 analytics-ready institutions.
- Aggregation tables have the expected sizes and grouping dimensions.
- ROI rankings are correctly sorted in descending order.
- No duplicate institutions were detected in the institution-level table.

With these checks complete, the Gold datasets are saved for use in the Power BI dashboard and Microsoft Fabric analytics environment.

---

### Saving Gold Layer Tables

The finalized Gold tables are exported for downstream analytics and dashboard development.  
These curated datasets will be used in Power BI and Microsoft Fabric to build interactive visualizations.

In [16]:
# Save Gold layer tables

gold_institution_roi.to_csv("../data_gold/gold_institution_roi.csv", index=False)
gold_state_summary.to_csv("../data_gold/gold_state_summary.csv", index=False)
gold_control_summary.to_csv("../data_gold/gold_control_summary.csv", index=False)
gold_locale_summary.to_csv("../data_gold/gold_locale_summary.csv", index=False)

print("Gold tables successfully saved.")

Gold tables successfully saved.
